# Corrélation et Régression Linéaire Simple
## World Happiness Report 2019-2024

Ce notebook explore la relation entre le bonheur des pays et ses déterminants socio-économiques à travers deux outils complémentaires :

- La **corrélation** — mesurer la force et la direction d'une relation linéaire entre deux variables
- La **régression linéaire simple** — modéliser cette relation pour prédire une variable cible à partir d'une variable explicative

Le dataset utilisé est le **World Happiness Report 2025** (données 2019-2024), publié annuellement par le Sustainable Development Solutions Network. Il couvre ~150 pays par année et décompose le score de bonheur (`life_ladder`) en contributions de plusieurs facteurs socio-économiques.

---

## Sommaire

- [1. Import des bibliothèques](#1)
- [2. Chargement et inspection des données](#2)
- [3. Nettoyage](#3)
- [4. Visualisation exploratoire](#4)
- [5. Corrélation](#5)
- [6. Régression linéaire simple](#6)
- [7. Diagnostic des résidus](#7)

---

## 1. Import des bibliothèques

Les bibliothèques utilisées dans ce notebook :

- `pandas` — manipulation des données
- `numpy` — calculs numériques
- `matplotlib` / `seaborn` — visualisation
- `scipy.stats` — calcul du coefficient de corrélation de Pearson
- `sklearn` — modèle de régression, découpage train/test, métriques d'évaluation

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn import metrics

## 2. Chargement et inspection des données

Les données sont chargées directement depuis GitHub — le fichier CSV est disponible à l'adresse suivante :

In [ ]:
url = 'https://raw.githubusercontent.com/cecile-learns/preparation_formation/main/Data/world_happiness.csv'
df=pd.read_csv(url)

On commence par une inspection rapide du dataset : dimensions, types de variables, premières et dernières lignes.

In [ ]:
print(df.shape)
print(df.dtypes)

In [ ]:
df.head()

In [ ]:
df.tail()

In [ ]:
# Doublons
print(df.duplicated().sum())

In [ ]:
df.describe()

**Observations** :
- Le dataset contient `[à compléter après exécution]` lignes et `[à compléter]` colonnes
- Les noms de colonnes sont longs et contiennent des espaces — ils seront renommés en partie 3
- La variable cible est `Life evaluation (3-year average)` — le score de bonheur perçu sur une échelle de 0 à 10
- `df.tail()` révèle des lignes entièrement vides en fin de fichier — artefact de l'export Excel. Elles seront supprimées en partie 3.
- Il y a 654 doublons qui pourraient être des valeurs nulles seulement, ce sera à confirmer
- Il ne semble pas y avoir de outliers

## 3. Nettoyage

Le nettoyage se fait en 4 étapes dans cet ordre :
1. Suppression des lignes entièrement vides
2. Renommage des colonnes
3. Suppression des colonnes inutiles pour l'analyse
4. Gestion des valeurs manquantes sur les variables explicatives

### 3.1 Suppression des lignes vides

In [ ]:
df = df.dropna(how='all')

### 3.2 Renommage des colonnes

Les noms originaux sont longs et contiennent des espaces — on les remplace par des noms courts en minuscules.

In [ ]:
df = df.rename(columns={
    'Life evaluation (3-year average)': 'life_ladder',
    'Explained by: Log GDP per capita': 'log_gdp',
    'Explained by: Social support': 'social_support',
    'Explained by: Healthy life expectancy': 'life_expectancy',
    'Explained by: Freedom to make life choices': 'freedom',
    'Explained by: Generosity': 'generosity',
    'Explained by: Perceptions of corruption': 'corruption',
    'Dystopia + residual': 'dystopia_residual',
    'Country name': 'country',
    'Lower whisker': 'lower_ci',
    'Upper whisker': 'upper_ci'
})

### 3.3 Suppression des colonnes inutiles

Quatre colonnes sont exclues de l'analyse :
- `lower_ci` et `upper_ci` — intervalles de confiance autour de `life_ladder`, méta-données sur la précision de la mesure
- `Rank` — classement calculé directement depuis `life_ladder` (redondant)
- `dystopia_residual` — composante calculée de `life_ladder` par construction dans le rapport WHR. L'inclure comme variable explicative serait une fuite de données

> ⚠️ **Fuite de données** : utiliser comme variable explicative une variable calculée à partir de la variable cible revient à prédire y avec une partie de y. Le modèle serait artificiellement performant mais sans valeur prédictive réelle.

In [ ]:
df = df.drop(columns=['lower_ci', 'upper_ci', 'Rank', 'dystopia_residual'])

### 3.4 Gestion des valeurs manquantes

Le rapport WHR ne publie les décompositions détaillées que pour les années 2019-2024. Les années antérieures ont des valeurs manquantes sur toutes les variables explicatives. On supprime les lignes incomplètes sur ces variables.

In [ ]:
df_clean = df.dropna(subset=['log_gdp', 'social_support', 'life_expectancy',
                              'freedom', 'generosity', 'corruption'])
print(df_clean.shape)
print(df_clean['Year'].value_counts().sort_index())
df_clean.info()

In [ ]:
print(df_clean.duplicated().sum())

> **Résultat** : le dataset nettoyé contient 868 lignes couvrant les années 2019-2024.

## 4. Visualisation exploratoire

Avant d'ajuster un modèle, on visualise les données pour identifier les relations potentielles entre variables. Le `pairplot` est un outil de premier tour : il trace simultanément les distributions univariées (diagonale) et les nuages de points bivariés (hors diagonale) pour toutes les variables numériques.

In [ ]:
sns.pairplot(df_clean.drop(columns=['Year', 'country']))
plt.show()

> **Observations** :
> - `log_gdp`, `social_support` et `life_expectancy` montrent des nuages allongés avec `life_ladder` — corrélations positives visibles
> - `freedom` montre un signal positif plus modéré
> - `generosity` et `corruption` : nuages éparpillés, relation peu lisible visuellement
> - Plusieurs variables explicatives semblent corrélées entre elles (`log_gdp` × `social_support`, `log_gdp` × `life_expectancy`) — point à confirmer en section 5

> ⚠️ Le pairplot est un outil d'exploration visuelle. Il permet d'identifier des tendances, pas de les quantifier. La section suivante chiffre ces observations avec le coefficient de corrélation de Pearson.

## 5. Corrélation

La corrélation de Pearson mesure la force et la direction d'une relation linéaire entre deux variables. Elle est comprise entre -1 et +1 :
- **r proche de 1** — relation linéaire positive forte
- **r proche de -1** — relation linéaire négative forte  
- **r proche de 0** — absence de relation linéaire

> ⚠️ r = 0 ne signifie pas indépendance — deux variables peuvent être fortement liées de façon non linéaire avec r = 0. Toujours visualiser le nuage de points avant de conclure.

### 5.1 Matrice de corrélation

In [ ]:
sns.heatmap(df_clean.drop(columns=['Year', 'country']).corr(),
            annot=True,
            cmap='coolwarm',
            vmin=-1, vmax=1,
            fmt='.2f')
plt.title('Matrice de corrélation — World Happiness 2019-2024')
plt.show()

### 5.2 Corrélation ciblée — life_ladder et ses déterminants

In [ ]:
correlations = df_clean.drop(columns=['Year', 'country']).corr()['life_ladder'].drop('life_ladder').sort_values(ascending=False)
print(correlations)

In [ ]:
for col in ['log_gdp', 'social_support', 'life_expectancy', 'freedom', 'generosity', 'corruption']:
    r, p = stats.pearsonr(df_clean[col], df_clean['life_ladder'])
    print(f"{col:20s} r = {r:.3f} | p = {p:.4f}")

> **Observations — matrice de corrélation et corrélations ciblées :**
> - `log_gdp` et `social_support` sont les variables les plus corrélées avec `life_ladder` (r = 0.69) — ex-aequo en tête
> - `life_expectancy` suit de près (r = 0.66), puis `freedom` (r = 0.54)
> - `corruption` affiche r = 0.43 — positif car la variable mesure la **contribution de la faible corruption** au score, pas un taux brut de corruption
> - `generosity` n'a pratiquement aucune relation linéaire avec `life_ladder` (r = 0.04) ni avec les autres variables — elle sera peu utile en régression
> - `log_gdp`, `social_support` et `life_expectancy` sont corrélées entre elles (r entre 0.51 et 0.62) — elles partagent une information commune, signe de multicolinéarité potentielle en régression multiple
> - **`log_gdp` est retenue comme variable explicative pour la régression simple** — r = 0.69, signal le plus fort et interprétation économique claire

## 6. Régression linéaire simple

La régression linéaire simple modélise la relation entre une variable explicative `x` et une variable cible `y` par une droite :ŷ = w × x + b
où `w` est la pente (coefficient de régression) et `b` l'intercept (valeur de `y` quand `x` = 0).

Les paramètres `w` et `b` sont estimés en minimisant la somme des carrés des résidus (écarts entre valeurs réelles et valeurs prédites) — c'est la **méthode des moindres carrés**.

### 6.1 Découpage train/test

Avant d'ajuster le modèle, on divise le dataset en deux ensembles :
- **Ensemble d'entraînement (80%)** — utilisé pour estimer les paramètres du modèle
- **Ensemble de test (20%)** — utilisé pour évaluer les performances sur des données non vues

> Cette séparation est fondamentale en ML : évaluer un modèle sur les données qui ont servi à l'entraîner donne une mesure optimiste et non représentative des performances réelles.

In [ ]:
X = df_clean[['log_gdp']]
y = df_clean[['life_ladder']]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Taille totale      : {len(X)} observations")
print(f"Ensemble train     : {len(X_train)} observations ({len(X_train)/len(X)*100:.0f}%)")
print(f"Ensemble test      : {len(X_test)} observations ({len(X_test)/len(X)*100:.0f}%)")

### 6.2 Ajustement du modèle

In [ ]:
lr = LinearRegression().fit(X_train, y_train)

w = lr.coef_[0][0]
b = lr.intercept_[0]
print(f"Pente  w = {w:.3f}")
print(f"Intercept b = {b:.3f}")
print(f"Équation : life_ladder = {w:.3f} × log_gdp + {b:.3f}")

### 6.3 Prédictions et évaluation

In [ ]:
y_test_preds = lr.predict(X_test)

mae = metrics.mean_absolute_error(y_test, y_test_preds)
r2 = metrics.r2_score(y_test, y_test_preds)

print(f"MAE (erreur absolue moyenne) : {mae:.3f}")
print(f"R²                           : {r2:.3f}")

> **Interprétation** :
> - **MAE** — en moyenne, le modèle se trompe de `0,626 points sur l'échelle de bonheur (0-10)
> - **R²** — le modèle explique 48% de la variance de `life_ladder` à partir de `log_gdp` seul

### 6.4 Visualisation

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Droite de régression
axes[0].scatter(X_train, y_train, alpha=0.5, label='Entraînement')
axes[0].scatter(X_test, y_test_preds, alpha=0.5, color='red', label='Prédictions test')
axes[0].plot(X_test.sort_values('log_gdp'),
             lr.predict(X_test.sort_values('log_gdp')),
             color='red', lw=2)
axes[0].set_xlabel('log_gdp')
axes[0].set_ylabel('life_ladder')
axes[0].set_title(f'Régression linéaire simple\nR² = {r2:.3f}')
axes[0].legend()

# Valeurs réelles vs prédictions
axes[1].scatter(y_test, y_test_preds, alpha=0.5)
axes[1].plot([y_test.min(), y_test.max()],
             [y_test.min(), y_test.max()],
             color='red', linestyle='--', lw=2)
axes[1].set_xlabel('Valeurs réelles')
axes[1].set_ylabel('Valeurs prédites')
axes[1].set_title('Réel vs Prédit\n(droite rouge = modèle parfait)')

plt.tight_layout()
plt.show()

## 7. Diagnostic des résidus

Un bon modèle de régression linéaire produit des résidus (écarts entre valeurs réelles et prédites) qui respectent trois conditions :
- **Centrés sur zéro** — pas de biais systématique
- **Sans pattern** — si les résidus suivent une structure, la relation n'est pas linéaire
- **Distribution gaussienne** — hypothèse du modèle linéaire

In [ ]:
residus = y_test.values.flatten() - y_test_preds.flatten()

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Résidus vs prédictions
axes[0].scatter(y_test_preds, residus, alpha=0.5)
axes[0].axhline(0, color='red', linestyle='--')
axes[0].set_xlabel('Valeurs prédites')
axes[0].set_ylabel('Résidus')
axes[0].set_title('Résidus vs prédictions\n(pas de pattern = bon signe)')

# Distribution des résidus
axes[1].hist(residus, bins=20, density=True)
axes[1].set_xlabel('Résidu')
axes[1].set_title('Distribution des résidus\n(doit être gaussienne)')

# Résidus vs variable explicative
axes[2].scatter(X_test, residus, alpha=0.5)
axes[2].axhline(0, color='red', linestyle='--')
axes[2].set_xlabel('log_gdp')
axes[2].set_ylabel('Résidus')
axes[2].set_title('Résidus vs log_gdp')

plt.tight_layout()
plt.show()

- Résidus vs prédictions (gauche) : pas de pattern en éventail ni en courbe — homoscédasticité respectée. Les résidus sont bien répartis de part et d'autre de zéro.

- Distribution des résidus (centre) : approximativement gaussienne, centrée sur zéro. Les valeurs extrêmes à -3 sont des outliers réels — pays dont le bonheur s'écarte fortement de ce que le PIB prédit. Acceptable.

- Résidus vs log_gdp (droite) : pas de pattern visible, variance stable sur toute la plage de log_gdp.

**Conclusion diagnostic** : les hypothèses du modèle linéaire sont globalement respectées. Le modèle est valide.